In [0]:
RAW_PATH='/Volumes/workspace/default/eccomerce/'
BRONZE_PATH='/Volumes/workspace/default/eccomerce/bronze/'
SILVER_PATH='/Volumes/workspace/default/eccomerce/silver/'
GOLD_PATH='/Volumes/workspace/default/eccomerce/gold/'

#KEY OF GCP PROJECT
GCP_PROJECT='decisive-cinema-489218-g0'
BQ_QUERY='ecommerce'
TEMP_GCS_BUCKET='ecommerce-databricks-temp'


#secrets codes
GCP_SECRET_SCOPE='gcp-secret'
GCP_SECRET_KEY='gcp-sa-key'

In [0]:
from pyspark.sql.functions import sum as _sum, countDistinct, avg, desc

silver=spark.read.format('delta').load(SILVER_PATH+'transactions_enriched')
silver.createOrReplaceTempView('silver_txn')

In [0]:
gold_daily_store_cat=spark.sql("""
                               select transaction_date, store_name, 
                               location as store_location, category, 
                               sum(total_amount) as gross_sales, 
                               sum(final_amount) as net_sales, 
                               count(distinct customer_id) as unique_customer,
                               count(*) as txn_count
                               from silver_txn
                               group by transaction_date, store_name, store_location, category
                               """)
#save data
gold_daily_store_cat.write.mode('overwrite').partitionBy('transaction_date').format('delta').save(GOLD_PATH+'daily_store_category')

In [0]:
gold_top_customer=spark.sql("""
                            select customer_id, customer_name, store_name,region,
                            sum(final_amount) as total_spent, count(*) as txn_count
                            from silver_txn
                            group by customer_id, customer_name, store_name, region
                            order by total_spent desc
                            """)
gold_top_customer.write.mode('overwrite').format('delta').save(GOLD_PATH+'top_customers')

In [0]:
gold_promo=spark.sql("""
                     select promotion_id, count(*) as promo_txns,
                     sum(final_amount) as promo_sales,
                     avg(final_amount) as avg_with_promo
                     from silver_txn
                     where promotion_id is not null
                     group by promotion_id
                     """)
gold_promo.write.mode('overwrite').format('delta').save(GOLD_PATH+'promo_impact')

In [0]:
gold_sentiment=spark.sql("""
                         select product_id, product_name, category, avg(rating) as avg_rating,
                         count(rating) as rating_count
                         from silver_txn
                         where rating is not null
                         group by product_id, product_name, category
                         """)
gold_sentiment.write.mode('overwrite').format('delta').save(GOLD_PATH+'product_sentiment')


In [0]:
from pyspark.sql.functions import max as _max, count as _count, sum as _sum, datediff, current_date, to_timestamp, when, col

rfm=(silver.groupBy('customer_id', 'customer_name')
     .agg(
         _max('transaction_date').alias('last_txn'),
         _count('transaction_id').alias('frequency'),
         _sum('final_amount').alias('monetary')
         )
     .withColumn('recency_days', datediff(current_date(), to_timestamp('last_txn'))
     )

)
rfm=(rfm.withColumn('recency_bucket', when(col('recency_days')<=30, 'o-30')
                    .when(col('recency_days')<=90, '31-90')
                    .otherwise('90+'))
     .withColumn('frequency_bucket', when(col('frequency')>=10, 'high')
                    .when(col('frequency')>=3, 'medium')
                    .otherwise('low'))
     .withColumn('monetary_bucket', when(col('monetary')>=1000, 'high')
                    .when(col('monetary')>=300, 'medium')
                    .otherwise('low'))
                   
     )
rfm.write.mode('overwrite').format('delta').save(GOLD_PATH+'rfm')

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, unix_timestamp

w=Window.partitionBy('customer_id').orderBy('transaction_date')
tnx_with_prev=silver.withColumn('prev_txn', lag('transaction_date').over(w)).withColumn('prev_store', lag('store_id').over(w)) 
tnx_with_prev=tnx_with_prev.withColumn('time_dff_mins', (unix_timestamp('transaction_date') -unix_timestamp('prev_txn'))/(60))
suspects=tnx_with_prev.filter((col('time_dff_mins')<30) & (col('total_amount') >1000) & (col('store_id') != col('prev_store')))
suspects.write.mode('overwrite').format('delta').save(GOLD_PATH+'suspect_customers')